# Privacy-Preserving Federated Network Intrusion Detection System

## Notebook 03 - Data Preprocessing

### Objectives

- Load CICIDS2017 data
- Standardize feature names
- Handle missing and infinite values
- Remove duplicate records
- Clean invalid numerical values
- Encode the target labels
- Prepare the dataset for machine learning
- Save the processed dataset

In [1]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# =====================================================
# PROJECT PATHS
# =====================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"Raw data     : {RAW_DATA_PATH}")
print(f"Processed data: {PROCESSED_DATA_PATH}")

Raw data     : c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\raw\CICIDS2017
Processed data: c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed


In [3]:
# =====================================================
# LOAD AND MERGE DATASETS
# =====================================================

csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

dataframes = []

for file in csv_files:
    print(f"Loading: {file.name}")

    df = pd.read_csv(file, low_memory=False)

    dataframes.append(df)

merged_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nMerged dataset shape:", merged_df.shape)

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

Merged dataset shape: (2830743, 79)


In [4]:
# =====================================================
# CLEAN COLUMN NAMES
# =====================================================

merged_df.columns = (
    merged_df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

print("Column names cleaned.")
print(merged_df.columns.tolist())

Column names cleaned.
['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets', 'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets', 'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean', 'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max', 'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Flow_Bytes_s', 'Flow_Packets_s', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min', 'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Bwd_PSH_Flags', 'Fwd_URG_Flags', 'Bwd_URG_Flags', 'Fwd_Header_Length', 'Bwd_Header_Length', 'Fwd_Packets_s', 'Bwd_Packets_s', 'Min_Packet_Length', 'Max_Packet_Length', 'Packet_Length_Mean', 'Packet_Length_Std', 'Packet_Length_Variance', 'FIN_Flag_Count', 'SYN_Flag_Count', 'RST_Flag_Count', 'PSH_Flag_Count', 'ACK_Flag_Count', 'URG_Flag_Count', 'CWE_Flag_C

In [5]:
# =====================================================
# IDENTIFY TARGET LABEL
# =====================================================

label_column = merged_df.columns[-1]

print("Target column:", label_column)
print("\nClasses:")
print(merged_df[label_column].value_counts())

Target column: Label

Classes:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:
# =====================================================
# HANDLE INFINITE VALUES
# =====================================================

numeric_columns = merged_df.select_dtypes(
    include=np.number
).columns

merged_df[numeric_columns] = merged_df[numeric_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite values converted to NaN.")

Infinite values converted to NaN.


In [7]:
# =====================================================
# MISSING VALUE ANALYSIS
# =====================================================

missing_counts = merged_df.isnull().sum()

missing_counts = missing_counts[
    missing_counts > 0
].sort_values(ascending=False)

print("Columns containing missing values:")
print(missing_counts)

Columns containing missing values:
Flow_Bytes_s      2867
Flow_Packets_s    2867
dtype: int64


In [8]:
# =====================================================
# REMOVE MISSING VALUES
# =====================================================

before_missing = len(merged_df)

merged_df = merged_df.dropna()

after_missing = len(merged_df)

print(f"Rows before removing NaN : {before_missing:,}")
print(f"Rows after removing NaN  : {after_missing:,}")
print(f"Rows removed             : {before_missing - after_missing:,}")

Rows before removing NaN : 2,830,743
Rows after removing NaN  : 2,827,876
Rows removed             : 2,867


In [9]:
# =====================================================
# REMOVE DUPLICATES
# =====================================================

before_duplicates = len(merged_df)

merged_df = merged_df.drop_duplicates()

after_duplicates = len(merged_df)

print(f"Rows before duplicates removal : {before_duplicates:,}")
print(f"Rows after duplicates removal  : {after_duplicates:,}")
print(f"Duplicates removed             : {before_duplicates - after_duplicates:,}")

Rows before duplicates removal : 2,827,876
Rows after duplicates removal  : 2,520,798
Duplicates removed             : 307,078


In [10]:
# =====================================================
# CLEAN TARGET LABELS
# =====================================================

merged_df[label_column] = (
    merged_df[label_column]
    .astype(str)
    .str.strip()
)

merged_df = merged_df[
    merged_df[label_column] != ""
]

print("Remaining classes:")
print(merged_df[label_column].value_counts())

Remaining classes:
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [11]:
# =====================================================
# LABEL ENCODING
# =====================================================

label_encoder = LabelEncoder()

merged_df["Label_Encoded"] = label_encoder.fit_transform(
    merged_df[label_column]
)

label_mapping = pd.DataFrame({
    "Encoded_Label": range(len(label_encoder.classes_)),
    "Original_Label": label_encoder.classes_
})

label_mapping

,Encoded_Label,Original_Label
0,0,BENIGN
1,1,Bot
2,2,DDoS
3,3,DoS GoldenEye
4,4,DoS Hulk
5,5,DoS Slowhttptest
6,6,DoS slowloris
7,7,FTP-Patator
8,8,Heartbleed
9,9,Infiltration


In [12]:
# =====================================================
# PREPARE FEATURES AND TARGET
# =====================================================

X = merged_df.drop(
    columns=[label_column, "Label_Encoded"]
)

y = merged_df["Label_Encoded"]

print("Features:", X.shape)
print("Target :", y.shape)

Features: (2520798, 78)
Target : (2520798,)


In [13]:
# =====================================================
# KEEP NUMERICAL FEATURES
# =====================================================

X = X.select_dtypes(include=np.number)

print("Numerical feature count:", X.shape[1])

Numerical feature count: 78


In [14]:
# =====================================================
# HANDLE EXTREME VALUES
# =====================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.dropna()

y = y.loc[X.index]

print("Feature shape after cleaning:", X.shape)

Feature shape after cleaning: (2520798, 78)


In [15]:
# =====================================================
# FEATURE SCALING
# =====================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

print("Feature scaling completed.")
print(X_scaled.head())

Feature scaling completed.
   Destination_Port  Flow_Duration  Total_Fwd_Packets  Total_Backward_Packets  \
0          2.428596      -0.470914          -0.010425               -0.010950   
1          2.438537      -0.470911          -0.011684               -0.010003   
2          2.438590      -0.470913          -0.011684               -0.010003   
3          1.974744      -0.470913          -0.011684               -0.010003   
4          2.428491      -0.470914          -0.010425               -0.010950   

   Total_Length_of_Fwd_Packets  Total_Length_of_Bwd_Packets  \
0                    -0.056662                    -0.007566   
1                    -0.057228                    -0.007563   
2                    -0.057228                    -0.007563   
3                    -0.057228                    -0.007563   
4                    -0.056662                    -0.007566   

   Fwd_Packet_Length_Max  Fwd_Packet_Length_Min  Fwd_Packet_Length_Mean  \
0              -0.297774        

In [16]:
# =====================================================
# CREATE FINAL PROCESSED DATASET
# =====================================================

processed_df = X_scaled.copy()

processed_df["Label"] = y

print("Final processed dataset:")
print(processed_df.shape)

processed_df.head()

Final processed dataset:
(2520798, 79)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,2.428596,-0.470914,-0.010425,-0.010950,-0.056662,-0.007566,-0.297774,-0.217169,-0.294064,-0.260452,...,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764,0
1,2.438537,-0.470911,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764,0
2,2.438590,-0.470913,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764,0
3,1.974744,-0.470913,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764,0
4,2.428491,-0.470914,-0.010425,-0.010950,-0.056662,-0.007566,-0.297774,-0.217169,-0.294064,-0.260452,...,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764,0


In [17]:
# =====================================================
# SAVE PROCESSED DATASET
# =====================================================

output_file = (
    PROCESSED_DATA_PATH /
    "CICIDS2017_processed.csv"
)

processed_df.to_csv(
    output_file,
    index=False
)

print(f"Processed dataset saved to:\n{output_file}")

Processed dataset saved to:
c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed\CICIDS2017_processed.csv


In [18]:
# =====================================================
# SAVE LABEL MAPPING
# =====================================================

label_mapping.to_csv(
    PROCESSED_DATA_PATH / "label_mapping.csv",
    index=False
)

print("Label mapping saved.")

Label mapping saved.


In [19]:
# =====================================================
# FINAL VERIFICATION
# =====================================================

print("=" * 60)
print("FINAL DATASET VERIFICATION")
print("=" * 60)

print(f"Rows       : {processed_df.shape[0]:,}")
print(f"Features   : {processed_df.shape[1] - 1}")
print(f"Target     : Label")
print(f"Missing    : {processed_df.isnull().sum().sum():,}")
print(f"Duplicates : {processed_df.duplicated().sum():,}")

print("\nClass distribution:")
print(processed_df["Label"].value_counts().sort_index())

FINAL DATASET VERIFICATION
Rows       : 2,520,798
Features   : 78
Target     : Label
Missing    : 0
Duplicates : 6

Class distribution:
Label
0     2095057
1        1948
2      128014
3       10286
4      172846
5        5228
6        5385
7        5931
8          11
9          36
10      90694
11       3219
12       1470
13         21
14        652
Name: count, dtype: int64
